<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step02aDetectTrackTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02a · Detect and track cows: 1-minute test

Runs YOLO26n (COCO "cow") + ByteTrack on the **first minute** of `ReferenceClip.mp4`, at 15 fps, with a real-time preview. Use it to check settings quickly (~1 min) before the full run in Step 02b.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02aTest/`, **overwriting** each run:

- `Tracks.csv`: one row per box per processed frame (Frame, TimeSeconds, TrackId, X1..Y2, Confidence)
- `TrackSummary.csv`: one row per track
- `PreviewTracks.mp4`: the clip with boxes and TrackIds drawn; watch this to judge the result

A TrackId is not an AnimalId; that comes in later steps. Videos stay on Drive, never in git.
Needs a GPU: **Runtime > Change runtime type > T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.DetectTrack import DetectAndTrack, EstimateRealFps
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/   # step 01 output
ClipPath = VideoOutputDir / "ReferenceClip.mp4"
OutputDir = VideoOutputDir / "Step02aTest"

TargetFps = 15         # frames per second processed (clip is 30 fps)
Confidence = 0.25      # lower = more boxes (and more false ones)
MaxSeconds = 60        # only the first minute
PreviewSeconds = None  # preview of everything processed
PreviewSpeed = 1       # real time

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: detect + track, then show the summary and the preview video.
import pandas as pd
from IPython.display import Video, display

print(EstimateRealFps(ClipPath))   # share of repeated frames (high = the camera updates slowly)
Stats = DetectAndTrack(ClipPath, OutputDir, TargetFps=TargetFps, Confidence=Confidence, MaxSeconds=MaxSeconds,
                       PreviewSeconds=PreviewSeconds, PreviewSpeed=PreviewSpeed)

display(pd.read_csv(OutputDir / "TrackSummary.csv"))
PreviewPath = OutputDir / "PreviewTracks.mp4"
PreviewMb = PreviewPath.stat().st_size / 2**20
print(f"Preview: {PreviewPath} ({PreviewMb:.0f} MB)")
if PreviewMb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
    display(Video(str(PreviewPath), embed=True, width=960))

## About this step

**Models:** **YOLO26n** (Ultralytics, trained on COCO; only class 19 "cow" is kept) draws a box around each cow in a frame. **ByteTrack** links boxes across frames into *tracks*.

**How to read the output:** a `TrackId` is one stretch where the tracker followed the same box; it is **not a cow**. If the detector loses her for more than 10 s, the next box gets a new TrackId.

**What we found (first minute of the reference clip):** one cow, followed for the whole minute as track 1 (623 of 900 frames, confidence 0.71). A short, weak second track (10 frames, 0.31) was a second box on the same cow. ~93 % of consecutive frames are identical: the camera updates the picture only a few times a second.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.